# NetCDF to DGGS (H3) Conversion

This notebook provides an approach for converting NetCDF climate data to multi-resolution DGGS H3 format.

## Overview

This notebook processes NetCDF climate data to multi-resolution H3 DGGS format for use with [`pydggsapi`](https://github.com/LandscapeGeoinformatics/pydggsapi):

1. Determines optimal H3 resolution from NetCDF grid spacing
2. Processes NetCDF climate data to H3 DGGS format using `xarray` operations
3. Aggregates spatial data from lat/lon grid to H3 cells via `groupby` mean
4. Generates parent resolutions through cascading H3 aggregation (base → 0)
5. Combines monthly files into continuous timeseries via time concatenation
6. Creates hierarchical zarr with resolution groups (`res0`, `res1`, ..., `res6`)
7. Applies chunking for efficient data access (e.g.: chunks considering temporal montly/yearly splits)
8. Extracts variable metadata from NetCDF attributes and augment them with [ClimateData.ca](https://climatedata.ca/variables/) definitions
9. Generates complete `pydggsapi` configuration with spatiotemporal collection metadata

### Output Structure

After processing, your output directory will contain:
```
outputs/
├── dggs/
│   └── h3/
│       ├── res=0/                               # Coarsest resolution
│       │   └── {file}_h3_res=0.zarr
│       ├── res=1/
│       │   └── {file}_h3_res=1.zarr
│       ├── ...
│       ├── res=6/                               # Base resolution (finest)
│       │   └── {file}_h3_res=6.zarr
│       └── canada_climate_h3_dggs.zarr/         # Final combined hierarchical zarr
│           ├── .zmetadata                       # Consolidated metadata
│           ├── res0/                            # Group for resolution 0
│           │   ├── ssp126_prcptot_p10/          # Variable arrays
│           │   │   ├── .zarray                  # Array metadata (chunks, dtype, etc.)
│           │   │   └── 0.0, 0.1, ...            # Chunk files
│           │   └── ...
│           ├── res1/                            # Group for resolution 1
│           ├── ...
│           └── res6/                            # Group for resolution 6 (finest)
│               ├── ssp126_prcptot_p10/
│               │   ├── .zarray                  # chunks: (207466, 12) for monthly data
│               │   └── 0.0, 0.1, ..., 0.150     # 151 chunks (1 per year)
│               └── ...
├── pydggsapi_config.json                        # pydggsapi configuration
└── parent_resolutions_summary.json              # Parent aggregation stats
```

Hierarchical zarr structure details:
- Each resolution is stored as a separate group (`res0`, `res1`, ..., `res6`)
- Monthly data (`prcptot`, `tx_max`): Concatenated along time dimension
  - Shape: (N_cells, 1812) where 1812 = 151 years × 12 months
  - Chunks: (N_cells, 12) = 151 yearly chunks
  - Time coords: [1950-01-15, 1950-02-15, ..., 2100-12-15]
- Yearly data (`frost_days`, `ice_days`): Single time dimension
  - Shape: (N_cells, 151) where 151 = years 1950-2100
  - Chunks: (N_cells, 151) = 1 chunk
  - Time coords: [1950, 1951, ..., 2100]

### Cache Management

To reprocess data:
- Clear H3 grid cache: Remove files in `{CACHE_DIR}/h3_grid_*.npy`
- Clear metadata cache: Remove `{CACHE_DIR}/climate_variable_metadata.json`
- Clear processed zarr files: Remove directories in `{OUTPUT_DIR}/dggs/h3/res=*/`
- Clear final combined zarr: Remove `{FINAL_ZARR_PATH}`

---

## Technical Approach

**⚠️ Important Consideration ⚠️**

Both NetCDF and Zarr are chunked array formats. We leverage `xarray` to skip intermediate conversions with other formats.
However, we assume that the NetCDF files we are interested in are mostly aligned spatiotemporally already to skip certain steps.
For example, chucking will take into consideration that the time dimension is consistent across all files,
so the same chunking strategy applies identically for all DGGS resolutions to avoid fragmentation.

**Strategy**:
1. Compute `(lat, lon) → h3_id` mapping once per grid
2. Add H3 as coordinate to `xarray.Dataset`
3. Use `xarray`'s `groupby` to aggregate (time, lat, lon) → (time, h3_id)
4. Write directly to Zarr with optimal chunking

### Important Assumptions & Design Decisions

**Spatiotemporal Alignment**:
- NetCDF files are assumed to be mostly aligned spatiotemporally
- This allows skipping redundant alignment/resampling steps
- All files within an index/aggregation type share the same temporal grid

**Chunking Strategy**:
- **Spatial dimension (`h3_id`)**: Single chunk containing all H3 cells
  - Minimizes chunk overhead (metadata, compression headers)
  - Optimal for spatial queries (common in DGGS use cases)
  - Avoids fragmentation from `groupby()` operations

- **Temporal dimension (`time`)**: Single chunk containing all timesteps
  - Maintains consistency across all DGGS resolutions
  - Parent DGGS resolutions inherit temporal chunking from base resolution
  - Ensures uniform query performance across resolution hierarchy

**Why Chunking Matters**:
- Without explicit chunking, `groupby()` creates fragmented chunks (1000s of small chunks)
- Fragmentation causes massive storage bloat (~100x more storage for the same data) and poor read performance
- Small chunks have poor compression ratios and high metadata overhead
- Consistent chunking across resolutions enables predictable and efficient query performance


## Package Setup

### Installation
Only run this cell if packages are missing. Most should be available in the ogc-dggs environment.

In [1]:
%%capture
!pip install xarray xdggs zarr h3 numpy pandas tqdm beautifulsoup4 requests netcdf4

### Imports

In [2]:
import datetime
from pathlib import Path
from tqdm.auto import tqdm
import h3
import json
import numpy as np
import os
import pandas as pd
import requests
import time
import warnings
import xarray as xr
import zarr
from bs4 import BeautifulSoup

warnings.filterwarnings("ignore", message="Consolidated metadata is currently not part in the Zarr format 3 specification")

## Configuration

Define all paths and processing parameters.


In [3]:
def ensure_directory(path):
    """
    Ensure directory exists, creating it or following symlinks as needed.
    """
    path = Path(path)

    if path.is_symlink():
        target = path.resolve()
        target.mkdir(parents=True, exist_ok=True)
        return target
    else:
        path.mkdir(parents=True, exist_ok=True)
        return path


In [14]:
# Climate variables to process
CLIMATE_VARIABLES = ["prcptot", "tx_max"]  # ["prcptot", "tx_max", "ice_days", "frost_days"]

# Aggregation types: Monthly (MS) or Yearly (YS)
AGG_TYPE = "MS"

# Directory paths
BASE_DIR = Path(os.getenv("CLIMATE_DATA_BASE_DIR", "data/allyears"))
CACHE_DIR = ensure_directory(os.getenv("CLIMATE_CACHE_DIR", "cache"))
OUTPUT_DIR = ensure_directory(os.getenv("CLIMATE_OUTPUT_DIR", "outputs"))

# DGGS output hierarchy (consistent structure for all zarr outputs)
DGGS_OUTPUT_DIR = OUTPUT_DIR / "dggs"
H3_OUTPUT_DIR = DGGS_OUTPUT_DIR / "h3"

# Final output paths
FINAL_ZARR_PATH = H3_OUTPUT_DIR / "canada_climate_h3_dggs.zarr"
FINAL_CONFIG_PATH = OUTPUT_DIR / "pydggsapi_config.json"

print(f"Configuration loaded:")
print(f"  Climate variables: {CLIMATE_VARIABLES}")
print(f"  Aggregation type: {AGG_TYPE}")
print(f"  Base directory: {BASE_DIR}")
print(f"  Cache directory: {CACHE_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")
print(f"  DGGS output hierarchy: {H3_OUTPUT_DIR}")
print(f"  Final zarr output: {FINAL_ZARR_PATH}")
print(f"  Final config output: {FINAL_CONFIG_PATH}")


Configuration loaded:
  Climate variables: ['prcptot', 'tx_max']
  Aggregation type: MS
  Base directory: data/allyears
  Cache directory: /misc/scratch13/OGC11/canada-climate/cache
  Output directory: /misc/scratch13/OGC11/canada-climate/outputs
  DGGS output hierarchy: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3
  Final zarr output: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3/canada_climate_h3_dggs.zarr
  Final config output: /misc/scratch13/OGC11/canada-climate/outputs/pydggsapi_config.json


## Step 1: Determine Optimal H3 Resolution

Analyze NetCDF grid spacing and find the best matching H3 resolution.


In [5]:
def analyze_netcdf_resolution(nc_files):
    """
    Analyze spatial resolution of NetCDF files and determine optimal H3 resolution.

    Returns:
        tuple: (h3_resolution, grid_info_dict)
    """
    print(f"Analyzing {len(nc_files)} NetCDF files for spatial resolution...")

    resolutions = []
    for f in tqdm(nc_files, desc="Analyzing files"):
        ds = xr.open_dataset(f, decode_timedelta=False)
        lat = ds["lat"] if "lat" in ds else ds["latitude"]
        lon = ds["lon"] if "lon" in ds else ds["longitude"]

        # Estimate resolution (in degrees)
        lat_res = float(np.abs(lat[1] - lat[0]))
        lon_res = float(np.abs(lon[1] - lon[0]))

        # Approximate km using haversine formula
        mean_lat = float(lat.mean())
        earth_radius_km = 6371.0
        lat_km = lat_res * (np.pi/180) * earth_radius_km
        lon_km = lon_res * (np.pi/180) * earth_radius_km * np.cos(mean_lat * np.pi/180)

        resolutions.append((f, lat_km, lon_km))
        ds.close()

    # Find best (finest) resolution
    best_file, best_lat_km, best_lon_km = min(resolutions, key=lambda x: min(x[1], x[2]))
    print(f"\nBest spatial resolution: {best_lat_km:.2f} km x {best_lon_km:.2f} km")
    print(f"  from: {Path(best_file).name}")

    # Map NetCDF Grid to H3 DGGS
    # Find the finest H3 resolution where edge <= grid resolution
    h3_resolution = None
    for res in range(16):
        h3_edge_km = h3.average_hexagon_edge_length(res, "km")
        if h3_edge_km <= min(best_lat_km, best_lon_km):
            h3_resolution = res
            break

    if h3_resolution is None:
        h3_resolution = 15
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")
    else:
        h3_edge_km = h3.average_hexagon_edge_length(h3_resolution, "km")

    print(f"\nSelected H3 resolution: {h3_resolution} (edge ~{h3_edge_km:.3f} km)")
    print(f"  Reasoning: H3 res {h3_resolution} has edge {h3_edge_km:.3f} km <= grid {min(best_lat_km, best_lon_km):.2f} km")

    return h3_resolution, {
        'best_lat_km': best_lat_km,
        'best_lon_km': best_lon_km,
        'h3_edge_km': h3_edge_km,
        'best_file': best_file
    }


## Step 2: Helper Functions

In [6]:
def get_dggs_output_path(h3_resolution, filename=None):
    """
    Generate consistent output path for DGGS H3 zarr files.
    """
    output_dir = H3_OUTPUT_DIR / f"res={h3_resolution}"

    if filename:
        return output_dir / filename
    return output_dir


Compute H3 variables for entire lat/lon grid at once

In [7]:
def compute_h3_grid(lat, lon, resolution):
    """
    Compute H3 variables for a lat/lon grid (vectorized).
    Returns 2D array of H3 variables as int64.
    """
    lat_grid, lon_grid = np.meshgrid(lat, lon, indexing='ij')
    lat_flat = lat_grid.ravel()
    lon_flat = lon_grid.ravel()

    # Still need list comprehension for h3 library, but vectorized the grid creation
    h3_cells = np.array([
        int(h3.latlng_to_cell(float(la), float(lo), resolution), 16)
        for la, lo in zip(lat_flat, lon_flat)
    ], dtype=np.int64)

    return h3_cells.reshape(lat_grid.shape)


In [8]:
def build_zarr_encoding(dataset):
    """Return per-variable chunk metadata for writing to Zarr."""
    encoding = {}
    for var in dataset.data_vars:
        encoding[var] = {
            "chunks": tuple(dataset[var].shape)
        }
    return encoding


## Step 3: Process Single NetCDF File to H3 Zarr

In [9]:
def process_netcdf_to_h3_single_resolution(nc_file, h3_resolution, skip_existing=True):
    """
    Process NetCDF to H3 DGGS at single resolution using xarray operations.
    """
    timings = {}
    start_total = time.time()

    print(f"\nProcessing: {Path(nc_file).name}")

    # Create resolution-specific output directory using helper function
    output_dir = get_dggs_output_path(h3_resolution)
    output_dir.mkdir(parents=True, exist_ok=True)
    output_zarr = output_dir / f"{Path(nc_file).stem}_h3_res={h3_resolution}.zarr"

    # Check if already exists
    if skip_existing and output_zarr.exists():
        print(f"  ⏭️ Output already exists, skipping")
        ds_existing = xr.open_zarr(output_zarr, decode_timedelta=False)
        timings['total'] = 0
        timings['skipped'] = True
        return ds_existing, timings, output_zarr

    # Open NetCDF (no chunking yet, keep it simple)
    t0 = time.time()
    ds = xr.open_dataset(nc_file, decode_timedelta=False)
    timings['open_netcdf'] = time.time() - t0

    # Get latitude and longitude variables
    lat = ds['lat'].values if 'lat' in ds else ds['latitude'].values
    lon = ds['lon'].values if 'lon' in ds else ds['longitude'].values

    # Compute or load cached H3 grid mapping
    grid_hash = hash((tuple(lat), tuple(lon), h3_resolution))
    cache_file = Path(CACHE_DIR) / f"h3_grid_{grid_hash}.npy"

    t0 = time.time()
    if cache_file and cache_file.exists():
        print(f"  Loading cached H3 grid...")
        h3_grid = np.load(cache_file)
    else:
        print(f"  Computing H3 grid (res={h3_resolution})...")
        h3_grid = compute_h3_grid(lat, lon, h3_resolution)
        if cache_file:
            cache_file.parent.mkdir(parents=True, exist_ok=True)
            np.save(cache_file, h3_grid)
    timings['h3_grid'] = time.time() - t0

    # Add H3 index as a coordinate
    print(f"  Adding H3 coordinate to dataset...")
    t0 = time.time()
    ds = ds.assign_coords({'h3_index': (('lat', 'lon'), h3_grid)})
    timings['assign_coords'] = time.time() - t0

    # Stack spatial dimensions: (time, lat, lon) → (time, spatial)
    print(f"  Stacking spatial dimensions...")
    t0 = time.time()
    ds_stacked = ds.stack(spatial=['lat', 'lon'])
    timings['stack'] = time.time() - t0

    # Drop NaN values (ocean/missing data)
    print(f"  Dropping NaN values...")
    t0 = time.time()
    ds_stacked = ds_stacked.dropna(dim='spatial', how='all')
    timings['dropna'] = time.time() - t0

    # Group by H3 and aggregate: (time, spatial) → (time, h3_id)
    print(f"  Aggregating by H3 cell...")
    t0 = time.time()
    ds_h3 = ds_stacked.groupby('h3_index').mean()
    timings['groupby_aggregate'] = time.time() - t0

    # Rename dimension for clarity
    ds_h3 = ds_h3.rename({'h3_index': 'h3_id'})

    # Rechunk for efficient storage
    # Spatial: all h3 cells in one chunk for minimal overhead
    # Temporal: all timesteps in one chunk to maintain consistency across resolutions
    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
        'time': ds_h3.sizes['time'] if 'time' in ds_h3.sizes else -1
    }
    ds_h3 = ds_h3.chunk(optimal_chunks)

    encoding = build_zarr_encoding(ds_h3)

    print(f"  Writing to Zarr: {output_zarr}")
    t0 = time.time()
    ds_h3.to_zarr(output_zarr, mode='w', encoding=encoding, compute=True)
    timings['write_zarr'] = time.time() - t0

    ds.close()

    timings['total'] = time.time() - start_total

    print(f"  ⏱️ Total time: {timings['total']:.2f}s")
    print(f"     ├─ H3 grid: {timings['h3_grid']:.2f}s")
    print(f"     ├─ Stack/aggregate: {timings['stack'] + timings['groupby_aggregate']:.2f}s")
    print(f"     └─ Write Zarr: {timings['write_zarr']:.2f}s")

    return ds_h3, timings, output_zarr


## Step 4: Test with Actual Data

In [10]:
# Find test files
test_files = list(BASE_DIR.glob("**/*prcptot*.nc"))
if test_files:
    print(f"Found {len(test_files)} prcptot files")

    # Determine optimal H3 resolution from the data
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(test_files[:5])  # Analyze first 5 files

    print(f"\n{'='*60}")
    print(f"Will use H3 resolution: {H3_RESOLUTION}")
    print(f"{'='*60}\n")

    # Process first file as test
    test_file = test_files[0]
    print(f"Test file: {test_file}")

    # Process it (will skip if already exists)
    result, timings, output_path = process_netcdf_to_h3_single_resolution(
        test_file,
        H3_RESOLUTION,
        skip_existing=True
    )

    print(f"\n✅ Test completed successfully!")
    print(f"Result shape: {result.dims}")
    print(f"Variables: {list(result.data_vars)}")
    print(f"Output: {output_path}")
else:
    print("No test files found")


Found 17 prcptot files
Analyzing 5 NetCDF files for spatial resolution...


Analyzing files:   0%|          | 0/5 [00:00<?, ?it/s]


Best spatial resolution: 9.27 km x 4.31 km
  from: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Selected H3 resolution: 6 (edge ~3.725 km)
  Reasoning: H3 res 6 has edge 3.725 km <= grid 4.31 km

Will use H3 resolution: 6

Test file: data/allyears/prcptot/MS/prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
  ⏭️ Output already exists, skipping

✅ Test completed successfully!
Result shape: FrozenMappingWarningOnValuesAccess({'h3_id': 207466, 'time': 151})
Variables: ['ssp126_prcptot_p10', 'ssp126_prcptot_p50', 'ssp126_prcptot_p90', 'ssp245_prcptot_p10', 'ssp245_prcptot_p50', 'ssp245_prcptot_p90', 'ssp370_prcptot_p10', 'ssp370_prcptot_p50', 'ssp370_prcptot_p90', 'ssp585_prcptot_p10', 'ssp585_prcptot_p50', 'ssp585_prcptot_p90']
Output: /misc/scratch13/OGC11/canada-climate/outputs/dggs/h3/res=6/prcptot_

## Step 5: Process All Monthly NetCDF Files

In [11]:
def process_all_nc_files(nc_files, h3_resolution, cache_dir):
    """
    Process all NetCDF files.

    Args:
        nc_files: List of NetCDF file paths to process
        h3_resolution: H3 resolution to use
        cache_dir: Cache directory

    Returns:
        dict: Summary statistics and timing info
    """
    all_timings = []
    processed_files = []

    all_nc_files = sorted(set(nc_files))  # Remove duplicates

    print(f"\n{'='*70}")
    print(f"Processing {len(all_nc_files)} NetCDF files at H3 resolution {h3_resolution}")
    print(f"{'='*70}\n")

    start_total = time.time()

    for i, nc_file in enumerate(all_nc_files, 1):
        print(f"\n[{i}/{len(all_nc_files)}] {nc_file.name}")

        try:
            result, timings, output_path = process_netcdf_to_h3_single_resolution(
                nc_file,
                h3_resolution,
                skip_existing=True
            )

            # Only add timing if actually processed (not skipped)
            if not timings.get('skipped', False):
                all_timings.append({
                    'file': nc_file.name,
                    'variables': list(result.data_vars),
                    'n_variables': len(result.data_vars),
                    'n_cells': result.sizes.get('h3_id', 0),
                    'n_timesteps': result.sizes.get('time', 0),
                    **timings
                })

            processed_files.append(output_path)

        except Exception as e:
            print(f"  ❌ Error processing {nc_file.name}: {e}")
            continue

    total_time = time.time() - start_total

    print(f"\n{'='*70}")
    print(f"✅ Processing complete!")
    print(f"{'='*70}")
    print(f"Total time: {total_time/60:.2f} minutes ({total_time:.1f}s)")
    print(f"Processed: {len(processed_files)} files")
    print(f"Average time per file: {total_time/max(len(all_timings), 1):.2f}s")

    # Save timing summary
    timing_summary = {
        'h3_resolution': h3_resolution,
        'total_files': len(all_nc_files),
        'processed_files': len(processed_files),
        'total_time_seconds': total_time,
        'timings': all_timings,
        'timestamp': datetime.datetime.now().isoformat()
    }

    summary_file = Path(OUTPUT_DIR) / f"timing_summary_res={h3_resolution}.json"
    with open(summary_file, 'w') as f:
        json.dump(timing_summary, f, indent=2)

    print(f"Timing summary saved to: {summary_file}")

    return timing_summary


## Step 6: Execute Full Processing

Process all monthly files for all climate variables.


In [12]:
# Find all files matching variables and aggregation types
all_files = []
for var in CLIMATE_VARIABLES:
    pattern = BASE_DIR / var / AGG_TYPE / "*.nc"
    files = list(pattern.parent.glob("*.nc")) if pattern.parent.exists() else []
    all_files.extend(files)
all_files = sorted(set(all_files))

if all_files:
    # Determine H3 resolution by sampling first 10 files
    sample_size = min(10, len(all_files))
    print(f"\nDetermining optimal H3 resolution from {sample_size} sample file(s)...\n")
    H3_RESOLUTION, grid_info = analyze_netcdf_resolution(all_files[:sample_size])

    print(f"\n{'='*70}")
    print(f"CONFIGURATION")
    print(f"{'='*70}")
    print(f"Climate variables: {CLIMATE_VARIABLES}")
    print(f"Aggregation type: {AGG_TYPE}")
    print(f"H3 resolution: {H3_RESOLUTION}")
    print(f"Total files to process: {len(all_files)}")
    print(f"Output directory: {OUTPUT_DIR}")
    print(f"{'='*70}\n")

    # Process all files
    summary = process_all_nc_files(
        nc_files=all_files,
        h3_resolution=H3_RESOLUTION,
        cache_dir=CACHE_DIR
    )
else:
    print("No NetCDF files found!")



Determining optimal H3 resolution from 10 sample file(s)...

Analyzing 10 NetCDF files for spatial resolution...


Analyzing files:   0%|          | 0/10 [00:00<?, ?it/s]


Best spatial resolution: 9.27 km x 4.31 km
  from: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Selected H3 resolution: 6 (edge ~3.725 km)
  Reasoning: H3 res 6 has edge 3.725 km <= grid 4.31 km

CONFIGURATION
Climate variables: ['prcptot', 'tx_max']
Aggregation type: MS
H3 resolution: 6
Total files to process: 24
Output directory: /misc/scratch13/OGC11/canada-climate/outputs


Processing 24 NetCDF files at H3 resolution 6


[1/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January.nc
  ⏭️ Output already exists, skipping

[2/24] prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc

Processing: prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February.nc
  ⏭️ Output already exists, skipping

[3/24] prcptot_MS_MBCn+PCIC-Blend_historical+a

## Step 7: Aggregate to Parent DGGS Resolutions

In [13]:
def aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr):
    """
    Aggregate H3 data from higher resolution to a parent resolution.

    Args:
        source_zarr: Path to source Zarr store (higher resolution)
        parent_res: Target parent resolution (lower number = coarser)
        output_zarr: Output Zarr path

    Returns:
        xarray Dataset at parent resolution
    """
    print(f"  Aggregating to parent resolution {parent_res}...")

    # Open source data
    ds = xr.open_zarr(source_zarr, decode_timedelta=False)

    # Get current H3 IDs (int64)
    h3_ids = ds['h3_id'].values

    # Convert to parent H3 IDs
    print(f"    Converting {len(h3_ids)} cells to parent res...")
    parent_h3_ids = np.array([
        int(h3.cell_to_parent(h3.int_to_str(h3_id), parent_res), 16)
        for h3_id in h3_ids
    ], dtype=np.int64)

    # Add parent H3 ID as coordinate
    ds = ds.assign_coords({'h3_parent': ('h3_id', parent_h3_ids)})

    # Group by parent and aggregate
    print(f"    Grouping and aggregating...")
    ds_parent = ds.groupby('h3_parent').mean()
    ds_parent = ds_parent.rename({'h3_parent': 'h3_id'})

    # Rechunk for efficient storage, preserving temporal chunking from source
    # This ensures consistency across all resolutions
    source_time_chunks = ds.chunks.get('time', None) if hasattr(ds, 'chunks') else None

    optimal_chunks = {
        'h3_id': -1,  # Single chunk for all h3 cells
    }

    # Preserve time chunking from source if available, otherwise use all timesteps
    if 'time' in ds_parent.sizes:
        if source_time_chunks and len(source_time_chunks) > 0:
            # Use same time chunking as source
            optimal_chunks['time'] = source_time_chunks[0]
        else:
            # Fallback: all timesteps in one chunk
            optimal_chunks['time'] = ds_parent.sizes['time']

    ds_parent = ds_parent.chunk(optimal_chunks)

    encoding = build_zarr_encoding(ds_parent)

    print(f"    Writing to {output_zarr}")
    ds_parent.to_zarr(output_zarr, mode='w', encoding=encoding, compute=True)

    ds.close()

    return ds_parent

def process_all_parent_resolutions(base_resolution):
    """
    Process all parent resolutions from base_resolution down to 0.

    Args:
        base_resolution: Starting (finest) H3 resolution

    Returns:
        dict: Summary of processed resolutions
    """
    # Find source files using helper function
    source_path = get_dggs_output_path(base_resolution)
    source_files = sorted(source_path.glob(f"*_h3_res={base_resolution}.zarr"))

    print(f"\n{'='*70}")
    print(f"Aggregating {len(source_files)} variables to parent resolutions")
    print(f"Base resolution: {base_resolution} → Target: 0-{base_resolution-1}")
    print(f"{'='*70}\n")

    summary = {}

    for parent_res in range(base_resolution - 1, -1, -1):
        print(f"\n{'='*70}")
        print(f"Processing resolution {parent_res}")
        print(f"{'='*70}")

        res_start = time.time()
        # Use helper function for consistent output path
        output_dir = get_dggs_output_path(parent_res)
        output_dir.mkdir(parents=True, exist_ok=True)

        processed_count = 0

        for source_file in tqdm(source_files, desc=f"Res {parent_res}"):
            # Determine source for this resolution
            if parent_res == base_resolution - 1:
                # First parent level: aggregate from base resolution
                source_zarr = source_file
            else:
                # Subsequent levels: aggregate from previous parent using helper function
                prev_dir = get_dggs_output_path(parent_res + 1)
                source_zarr = prev_dir / source_file.name.replace(
                    f"_h3_res={base_resolution}.zarr",
                    f"_h3_res={parent_res + 1}.zarr"
                )

                if not source_zarr.exists():
                    print(f"⚠️ Warning: Source not found: {source_zarr}")
                    continue

            # Output file for this parent resolution
            output_zarr = output_dir / source_file.name.replace(
                f"_h3_res={base_resolution}.zarr",
                f"_h3_res={parent_res}.zarr"
            )

            if output_zarr.exists():
                print(f"⏭️ Skipping {source_file.name} (already processed)")
                continue

            try:
                aggregate_to_parent_resolution(source_zarr, parent_res, output_zarr)
                processed_count += 1
            except Exception as e:
                print(f"❌ Error processing {source_file.name} to res {parent_res}: {e}")
                continue

        res_time = time.time() - res_start
        summary[parent_res] = {
            'processed': processed_count,
            'time_seconds': res_time
        }

        print(f"\n✅ Resolution {parent_res} complete: {processed_count} variables in {res_time:.1f}s")

    return summary


## Step 8: Execute Parent Resolution Aggregation

In [15]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"Starting parent resolution aggregation from resolution {H3_RESOLUTION}")

    parent_summary = process_all_parent_resolutions(base_resolution=H3_RESOLUTION)

    print(f"\n{'='*70}")
    print(f"✅ All parent resolutions complete!")
    print(f"{'='*70}")

    for res, info in sorted(parent_summary.items(), reverse=True):
        print(f"Resolution {res}: {info['processed']} variables in {info['time_seconds']:.1f}s")

    # Save summary
    summary_file = OUTPUT_DIR / "parent_resolutions_summary.json"
    with open(summary_file, mode="w", encoding="utf-8") as f:
        json.dump(parent_summary, f, indent=2)
    print(f"\nSummary saved to: {summary_file}")
else:
    print("H3_RESOLUTION not defined. Run previous cells first.")


Starting parent resolution aggregation from resolution 6

Aggregating 25 variables to parent resolutions
Base resolution: 6 → Target: 0-5


Processing resolution 5


Res 5:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 4:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

...

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 1:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

Res 0:   0%|          | 0/25 [00:00<?, ?it/s]

⏭️ Skipping frost_days_YS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_01January_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_02February_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_03March_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_04April_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_05May_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentiles_06June_h3_res=6.zarr (already processed)
⏭️ Skipping prcptot_MS_MBCn+PCIC-Blend_historical+allssps_1950-2100_AllYears_percentil

## Step 9: Combine Individual Zarr Files into Hierarchical Structure

In [16]:
def combine_zarr_files_to_grouped_zarr(output_base_dir, h3_resolution, final_output_path):
    """
    Combine individual zarr files across all resolutions into a single hierarchical zarr
    with resolution-based groups for pydggsapi.

    Args:
        output_base_dir: Base directory containing dggs/h3/res={N}/ subdirectories
        h3_resolution: Maximum H3 resolution (base resolution)
        final_output_path: Path for final combined zarr store

    Returns:
        dict: Mapping of resolution to group names
    """
    print(f"\n{'='*70}")
    print(f"Combining zarr files into hierarchical zarr structure")
    print(f"{'='*70}\n")

    final_output_path = Path(final_output_path)
    base_path = Path(output_base_dir) / "dggs" / "h3"

    # Remove existing output if it exists
    if final_output_path.exists():
        import shutil
        shutil.rmtree(final_output_path)

    group_mapping = {}

    # Process each resolution from 0 to h3_resolution
    for res in range(h3_resolution + 1):
        res_dir = base_path / f"res={res}"

        if not res_dir.exists():
            print(f"⚠️ Resolution {res} directory not found: {res_dir}")
            continue

        zarr_files = sorted(res_dir.glob("*.zarr"))

        if not zarr_files:
            print(f"⚠️ No zarr files found for resolution {res}")
            continue

        print(f"\nProcessing resolution {res}: {len(zarr_files)} file(s)")

        # Open all zarr files for this resolution
        datasets = []
        for zarr_file in tqdm(zarr_files, desc=f"  Loading res {res}"):
            try:
                ds = xr.open_zarr(zarr_file, decode_timedelta=False)
                datasets.append(ds)
            except Exception as e:
                print(f"    ⚠️ Failed to open {zarr_file.name}: {e}")
                continue

        if not datasets:
            print(f"  ⚠️ No valid datasets for resolution {res}")
            continue

        # Combine all datasets for this resolution
        print(f"  Combining {len(datasets)} dataset(s)...")

        if len(datasets) == 1:
            combined_ds = datasets[0]
        else:
            # Concatenate along time dimension to combine monthly files
            # Monthly files have SAME variable names but DIFFERENT time coordinates
            # (January has Januaries 1950-2100, February has Februaries 1950-2100, etc.)
            # Concat combines them into continuous timeseries
            combined_ds = xr.concat(
                datasets,
                dim='time',
                combine_attrs='override',
                coords='all'
            )
            # Sort by time to ensure chronological order
            combined_ds = combined_ds.sortby('time')

        # Rechunk to ensure consistent chunking for this resolution
        # Spatial: all h3 cells in one chunk (no spatial locality in H3 integer ordering)
        # Temporal: chunk by year for monthly data (12 months), single chunk for yearly data
        time_size = combined_ds.sizes.get('time', 0)
        if time_size > 1000:
            # Monthly data (151 years × 12 months = 1812 timesteps)
            # Chunk by year (12 months) for efficient seasonal/annual queries
            time_chunk = 12
            print(f"  Detected monthly data: {time_size} timesteps → yearly chunks (12 months)")
        elif time_size > 100:
            # Yearly data (151 timesteps)
            # Keep as single chunk
            time_chunk = -1
            print(f"  Detected yearly data: {time_size} timesteps → single chunk")
        else:
            # Small dataset, single chunk
            time_chunk = -1

        optimal_chunks = {
            'h3_id': -1,  # Single chunk for all h3 cells at this resolution
            'time': time_chunk
        }
        print(f"  Rechunking to: {optimal_chunks}")
        combined_ds = combined_ds.chunk(optimal_chunks)
        encoding = build_zarr_encoding(combined_ds)

        # Save to zarr group
        group_name = f"res{res}"
        print(f"  Writing to group '{group_name}'...")
        print(f"    Variables: {len(combined_ds.data_vars)}")
        print(f"    H3 cells: {combined_ds.sizes.get('h3_id', 0):,}")
        print(f"    Timesteps: {combined_ds.sizes.get('time', 0)}")

        # Write to zarr using the group parameter
        # Use compute=True to ensure rechunking is applied and chunks are properly stored
        # Use consolidated=False here because we're writing multiple groups incrementally
        # Consolidation happens once at the end after ALL resolution groups are written
        mode = 'w' if res == 0 else 'a'
        combined_ds.to_zarr(
            final_output_path,
            group=group_name,
            mode=mode,
            consolidated=False,
            encoding=encoding,
            compute=True
        )

        group_mapping[str(res)] = group_name

        # Close individual datasets
        for ds in datasets:
            ds.close()
        combined_ds.close()

    # Consolidate metadata once at the end for all groups
    # This creates a single .zmetadata file with metadata for all resolution groups
    # Making subsequent reads much faster (single metadata read instead of per-group)
    print(f"\n{'='*70}")
    print(f"Consolidating metadata...")
    print(f"{'='*70}\n")
    zarr.consolidate_metadata(final_output_path)

    print(f"✅ Hierarchical zarr created with {len(group_mapping)} resolution group(s)")
    print(f"   Groups: {list(group_mapping.values())}")

    return group_mapping


## Step 10: Create Final Hierarchical Zarr

In [17]:
if 'H3_RESOLUTION' in globals() and H3_RESOLUTION is not None:
    print(f"\n{'='*70}")
    print(f"CREATING FINAL HIERARCHICAL ZARR")
    print(f"{'='*70}\n")

    group_mapping = combine_zarr_files_to_grouped_zarr(
        output_base_dir=OUTPUT_DIR,
        h3_resolution=H3_RESOLUTION,
        final_output_path=FINAL_ZARR_PATH
    )

    print(f"\n✅ Hierarchical zarr created: {FINAL_ZARR_PATH}")
    print(f"   Resolution groups: {list(group_mapping.values())}")

else:
    print("H3_RESOLUTION not defined. Run previous cells first.")



CREATING FINAL HIERARCHICAL ZARR


Combining zarr files into hierarchical zarr structure


Processing resolution 0: 25 file(s)


  Loading res 0:   0%|          | 0/25 [00:00<?, ?it/s]

  Combining 25 dataset(s)...
  Detected monthly data: 3775 timesteps → yearly chunks (12 months)
  Rechunking to: {'h3_id': -1, 'time': 12}
  Writing to group 'res0'...
    Variables: 36
    H3 cells: 9
    Timesteps: 3775


ValueError: Specified Zarr chunks encoding['chunks']=(9, 3775) for variable named 'ssp126_tx_max_p90' would overlap multiple Dask chunks. Please check the Dask chunks at position 1 and 2, on axis 1, they are overlapped on the same Zarr chunk in the region slice(None, None, None). Writing this array in parallel with Dask could lead to corrupted data. To resolve this issue, consider one of the following options: - Rechunk the array using `chunk()`. - Modify or delete `encoding['chunks']`. - Set `safe_chunks=False`. - Enable automatic chunks alignment with `align_chunks=True`.

## Step 11: Metadata Extraction Functions

In [ ]:
def extract_netcdf_metadata(zarr_store_path):
    """
    Extract metadata from zarr store attributes (inherited from NetCDF).

    Returns:
        dict: Metadata by variable name
    """
    print(f"\nExtracting metadata from zarr attributes...")

    metadata = {}

    zarr_root = zarr.open_group(zarr_store_path, mode='r')
    available_groups = sorted([g for g in zarr_root.group_keys()], reverse=True)

    if not available_groups:
        print(f"  ⚠️ No groups found in zarr store")
        return metadata

    first_group = available_groups[0]
    print(f"  Reading metadata from group: {first_group}")

    ds = xr.open_zarr(zarr_store_path, group=first_group, decode_timedelta=False)

    for var_name in ds.data_vars:
        var_attrs = ds[var_name].attrs

        metadata[var_name] = {
            "title": var_attrs.get("long_name", var_name),
            "description": var_attrs.get("description", ""),
            "x-ogc-unit": var_attrs.get("units", ""),
            "url": None
        }

        if "standard_name" in var_attrs:
            std_name = var_attrs["standard_name"]
            metadata[var_name]["x-ogc-definition"] = (
                f"http://cfconventions.org/Data/cf-standard-names/current/build/cf-standard-name-table.html#{std_name}"
            )

    ds.close()

    print(f"  Extracted metadata for {len(metadata)} variables")

    return metadata


## Step 12: Build Climate Variable URL Metadata

In [ ]:
def supplement_metadata_from_web(metadata):
    """
    Supplement metadata with ClimateData.ca URLs.

    Uses global CLIMATE_VARIABLE_URL_MAP and CLIMATE_VARIABLE_VARIATIONS
    from this step.
    """
    if 'CLIMATE_VARIABLE_URL_MAP' not in globals():
        print("⚠️  CLIMATE_VARIABLE_URL_MAP not defined")
        return metadata

    variable_map = CLIMATE_VARIABLE_URL_MAP
    variable_variations = CLIMATE_VARIABLE_VARIATIONS

    matched_count = 0

    for var_name in metadata:
        if metadata[var_name].get("url"):
            continue

        var_name_lower = var_name.lower()

        # Try exact match
        if var_name in variable_map:
            metadata[var_name]["url"] = variable_map[var_name]["url"]
            matched_count += 1
            continue

        # Try variation match
        if var_name_lower in variable_variations:
            config_var = variable_variations[var_name_lower]
            if config_var in variable_map:
                metadata[var_name]["url"] = variable_map[config_var]["url"]
                matched_count += 1
                continue

        # Try partial match
        best_match = None
        best_len = 0

        for config_var in CLIMATE_VARIABLES:
            config_var_lower = config_var.lower()
            if config_var_lower in var_name_lower:
                if len(config_var_lower) > best_len:
                    best_match = config_var
                    best_len = len(config_var_lower)

        if best_match and best_match in variable_map:
            metadata[var_name]["url"] = variable_map[best_match]["url"]
            matched_count += 1

    print(f"   Matched {matched_count}/{len(metadata)} variables to ClimateData.ca URLs")

    return metadata


In [ ]:
# Build climate variable URL metadata
print("="*70)
print("BUILDING CLIMATE VARIABLE URL METADATA")
print("="*70)

CLIMATE_VARIABLE_URL_MAP = {}

for var_name in CLIMATE_VARIABLES:
    slug = var_name.replace("_", "-")
    url = f"https://climatedata.ca/variable/{slug}/"
    CLIMATE_VARIABLE_URL_MAP[var_name] = {
        "url": url,
        "slug": slug
    }

# Create variations for matching
CLIMATE_VARIABLE_VARIATIONS = {}
for var_name in CLIMATE_VARIABLES:
    variations = [
        var_name,
        var_name.replace("_", "-"),
        var_name.replace("_", ""),
    ]
    for variation in variations:
        CLIMATE_VARIABLE_VARIATIONS[variation.lower()] = var_name

print(f"\n✅ Built URL metadata for {len(CLIMATE_VARIABLE_URL_MAP)} climate variables")

print(f"\nClimate variables:")
for var_name, info in CLIMATE_VARIABLE_URL_MAP.items():
    print(f"  {var_name:20s} → {info['url']}")


## Step 13: Generate pydggsapi Configuration

In [ ]:
if 'H3_RESOLUTION' not in globals() or H3_RESOLUTION is None:
    print("H3_RESOLUTION not defined. Run previous cells first.")
elif not FINAL_ZARR_PATH.exists():
    print(f"⚠️  Hierarchical zarr not found. Run Step 10 first!")
else:
    print(f"✅ Hierarchical zarr ready: {FINAL_ZARR_PATH}")
    # Verify groups
    import zarr as z
    zg = z.open_group(FINAL_ZARR_PATH, mode='r')
    groups = list(zg.group_keys())
    print(f"   Resolution groups: {groups}")